In [0]:
# CELL 1 — Sources and experiment configuration
demand_path = "development_042_silver_sandbox.demand_forecast.demand_forecast_results"
ai_attr_silver_path = "development_042_silver_sandbox.demand_forecast.game_attr_ai_silver"

FORECAST_MONTHS = 6  # One target: total units across calendar months 1–6, per revenue type.
TEST_FRACTION = 0.20
CAP_QUANTILE = 0.99
RANDOM_SEED = 42

MLFLOW_EXPERIMENT_PATH = None  # Default: /Users/<current_user>/AI_feature_extraction_forecasts
MLFLOW_RUN_NAME = None         # Default: xgb_ai_attributes_total6m_zero_defaults_v6
MLFLOW_LOG_DATA_SNAPSHOT = True

XGB_PARAMS = dict(
    objective="reg:squarederror", n_estimators=300, learning_rate=0.05,
    max_depth=3, min_child_weight=10, subsample=0.9, colsample_bytree=0.8,
    reg_lambda=10.0, tree_method="hist", random_state=RANDOM_SEED, n_jobs=4,
)
if type(FORECAST_MONTHS) is not int or FORECAST_MONTHS != 6:
    raise ValueError("This notebook predicts one six-month total; FORECAST_MONTHS must be 6.")
if not 0 < TEST_FRACTION < 1:
    raise ValueError("TEST_FRACTION must be between 0 and 1.")
if not 0 < CAP_QUANTILE <= 1:
    raise ValueError("CAP_QUANTILE must be between 0 and 1.")

# Tuning uses earlier TRAINING games only; final TEST games remain untouched.
# Trial 0 always evaluates XGB_PARAMS. Remaining candidates are random samples.
TUNE_HYPERPARAMETERS = True
TUNING_TRIALS = 20  # Maximum candidates per target, including the configured default.
TUNING_FOLDS = 3
TUNING_MIN_FOLDS = 2
TUNING_INITIAL_TRAIN_FRACTION = 0.50
TUNING_MIN_TRAIN_GROUPS = 20
TUNING_MIN_VALIDATION_GROUPS = 5
TUNING_METRIC = "RMSE"  # "RMSE" or "MAE"; validation gives each game equal weight.
TUNING_SEARCH_SPACE = {
    "n_estimators": [150, 300, 600],
    "learning_rate": [0.02, 0.05, 0.10],
    "max_depth": [2, 3, 4, 5],
    "min_child_weight": [3, 10, 20],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0, 25.0],
    "reg_alpha": [0.0, 0.5, 2.0],
    "gamma": [0.0, 1.0, 5.0],
}

for name in ["TUNING_TRIALS", "TUNING_FOLDS", "TUNING_MIN_FOLDS",
             "TUNING_MIN_TRAIN_GROUPS", "TUNING_MIN_VALIDATION_GROUPS"]:
    if type(globals()[name]) is not int or globals()[name] < 1:
        raise ValueError(f"{name} must be a positive integer.")
if not 2 <= TUNING_MIN_FOLDS <= TUNING_FOLDS:
    raise ValueError("Require 2 <= TUNING_MIN_FOLDS <= TUNING_FOLDS.")
if not 0 < TUNING_INITIAL_TRAIN_FRACTION < 1:
    raise ValueError("TUNING_INITIAL_TRAIN_FRACTION must be between 0 and 1.")
if TUNING_METRIC not in {"RMSE", "MAE"}:
    raise ValueError("TUNING_METRIC must be RMSE or MAE.")
if not TUNING_SEARCH_SPACE or any(not isinstance(v, (list, tuple)) or not v
                                 for v in TUNING_SEARCH_SPACE.values()):
    raise ValueError("TUNING_SEARCH_SPACE must contain nonempty lists of parameter values.")
reserved = {"monotone_constraints", "early_stopping_rounds", "callbacks"}
if reserved & (set(XGB_PARAMS) | set(TUNING_SEARCH_SPACE)):
    raise ValueError("Tune n_estimators instead of early stopping; month constraints do not apply to total targets.")


In [0]:
# CELL 2 — Load game attributes and observed demand targets
df_demand_forecast = spark.table(demand_path)
ai_attr = spark.table(ai_attr_silver_path)
SOURCE_TABLES = {"demand": demand_path, "attributes": ai_attr_silver_path}

display(df_demand_forecast)
display(ai_attr)


In [0]:
# CELL 3 — Clean attributes and build one eligible six-month total per game/type.
from pyspark.sql import functions as F, types as T

# ============================================================
# 1. Helpers
# ============================================================
NULL_NAMES = ["", "none", "null", "nan", "na", "n/a"]


def clean_name(column):
    value = F.regexp_replace(
        F.lower(F.trim(column.cast("string"))),
        r"\s+",
        " ",
    )
    return F.when(~value.isin(NULL_NAMES), value)


def numeric_value(df, column_name, preferred_fields):
    """Extract a numeric scalar from a scalar or nested AI field."""
    dtype = df.schema[column_name].dataType

    if isinstance(dtype, T.StructType):
        fields = dtype.fieldNames()
        selected = next(
            (name for name in preferred_fields if name in fields),
            None,
        )

        # Accommodate alternate extraction schemas.
        if selected is None:
            numeric_fields = [
                field.name
                for field in dtype.fields
                if isinstance(field.dataType, T.NumericType)
            ]
            if len(numeric_fields) == 1:
                selected = numeric_fields[0]

        if selected is None:
            raise ValueError(
                f"Cannot identify the numeric field in {column_name}: "
                f"{fields}. Add its field name to preferred_fields."
            )

        value = F.col(column_name).getField(selected)
    else:
        value = F.col(column_name)

    # try_cast tolerates malformed extraction values.
    value = value.try_cast("double")
    return F.when(
        value.isNotNull()
        & ~F.isnan(value)
        & (F.abs(value) != F.lit(float("inf"))),
        value,
    )


def require_unique(df, keys, description):
    duplicates = (
        df.groupBy(*keys)
        .count()
        .where(F.col("count") > 1)
    )
    if duplicates.limit(1).count():
        display(duplicates.limit(20))
        raise ValueError(f"{description}: duplicated keys {keys}.")


# ============================================================
# 2. Clean AI attributes: one row per game
# ============================================================
theme_columns = sorted(
    name for name in ai_attr.columns
    if name.startswith("theme_") and name.endswith("_score")
)

mechanic_columns = sorted(
    name for name in ai_attr.columns
    if name.startswith("mechanic_") and name.endswith("_present")
)

# Use the curated key. Keep original names as metadata.
attrs = (
    ai_attr
    .withColumn("game_key", clean_name(F.col("join_game_name_key")))
    .where(F.col("game_key").isNotNull())
    .dropDuplicates()
)

from pyspark.sql.window import Window

game_row_window = Window.partitionBy("game_key").orderBy(
    F.when(
        F.lower(F.trim(F.col("parse_status"))) == "valid_json", 1
    ).otherwise(0).desc(),
    F.col("saved_at").desc_nulls_last(),
    F.col("run_id").desc_nulls_last(),
    F.col("game_name").asc_nulls_last(),
)

attrs = (
    attrs
    .withColumn("_row_number", F.row_number().over(game_row_window))
    .where(F.col("_row_number") == 1)
    .drop("_row_number")
)
attrs = (
    attrs
    .withColumn(
        "reels",
        numeric_value(
            attrs, "number_of_reels",
            ["value", "count", "number_of_reels", "number"],
        ),
    )
    .withColumn(
        "paylines_or_ways",
        numeric_value(
            attrs, "paylines_or_ways_count",
            ["value", "count", "paylines_or_ways_count", "number"],
        ),
    )
    .withColumn(
        "persistence_score",
        numeric_value(
            attrs, "persistence",
            ["score", "value", "persistence_score"],
        ),
    )
    .withColumn(
        "perceived_persistence_score",
        numeric_value(
            attrs, "perceived_persistence",
            ["score", "value", "perceived_persistence_score"],
        ),
    )
)

# Zero paylines/ways is a valid value (for example, a poker game).
# Missing semantic attributes are filled with zero below. Learn caps in training.
attrs = attrs.withColumn("reels", F.when(F.col("reels") > 0, F.col("reels")))
attrs = attrs.withColumn(
    "paylines_or_ways",
    F.when(F.col("paylines_or_ways") >= 0, F.col("paylines_or_ways")),
)

# Validate expected score ranges before applying the missing-as-zero rule.
attrs = (
    attrs
    .withColumn(
        "persistence_score",
        F.when(
            F.col("persistence_score").between(0, 1),
            F.col("persistence_score"),
        ),
    )
    .withColumn(
        "perceived_persistence_score",
        F.when(
            F.col("perceived_persistence_score").between(0, 5),
            F.col("perceived_persistence_score"),
        ),
    )
)

# User-defined semantics: missing paylines, persistence, themes, and mechanics
# represent absence (0), regardless of parse_status. Valid positive values remain.
ZERO_DEFAULT_NUMERIC_FEATURES = [
    "paylines_or_ways", "persistence_score", "perceived_persistence_score",
] + theme_columns + mechanic_columns

for name in theme_columns:
    value = F.col(name).try_cast("double")
    attrs = attrs.withColumn(name, F.when(value.between(0, 5), value))

for name in mechanic_columns:
    value = F.col(name).try_cast("double")
    attrs = attrs.withColumn(name, F.when(value.isin(0.0, 1.0), value))

attrs = attrs.fillna(0.0, subset=ZERO_DEFAULT_NUMERIC_FEATURES)

# A self-parent entry is not a separate parent game.
parent_key = clean_name(F.col("parent_game"))
original_name_key = clean_name(
    F.regexp_replace(
        F.lower(F.col("game_name")),
        r"\s+overview(?:-.*)?$",
        "",
    )
)

attrs = attrs.withColumn(
    "has_parent_game",
    F.when(parent_key.isNull(), F.lit(None).cast("double"))
    .when(
        (parent_key == F.col("game_key"))
        | (parent_key == original_name_key),
        F.lit(0.0),
    )
    .otherwise(F.lit(1.0)),
)

categorical_features = ["game_category", "game_classification"]

for name in categorical_features:
    attrs = attrs.withColumn(name, clean_name(F.col(name)))

numeric_features = [
    "reels",
    "paylines_or_ways",
    "persistence_score",
    "perceived_persistence_score",
    "has_parent_game",
] + theme_columns + mechanic_columns

metadata_columns = [
    "game_key",
    "game_name",
    "game_release_date",
    "to_sales_name_match",
    "to_rental_name_match",
]

ai_attr_clean_df = attrs.select(
    *metadata_columns,
    *numeric_features,
    *categorical_features,
)


# ============================================================
# 3. Reconstruct six CALENDAR months, then aggregate one target
# ============================================================
def prepare_six_month_targets(demand_df, horizon=6):
    """Use actuals only. Later observations confirm zero-activity interior gaps.

    Sales input is already cumulative: forward-fill it, then take month 6.
    Rental input is monthly: missing interior months are zero, then SUM 1–6.
    Neither padded curve rows nor imputed values establish coverage.
    """
    revenue = F.lower(F.trim(F.col("revenue_type")))
    rows = (
        demand_df
        .withColumn("game_key", clean_name(F.col("target_game")))
        .withColumn("target_type",
                    F.when(revenue.isin("sale", "sales"), "sales")
                    .when(revenue.isin("lease", "rental", "rent"), "rental"))
        .withColumn("month_index", F.col("month_index").try_cast("int"))
        .withColumn("raw_actual",
                    F.when(F.col("target_type") == "sales",
                           F.col("actual_cumulative_sales").try_cast("double"))
                    .otherwise(F.col("actual_monthly_lease").try_cast("double")))
        .withColumn("source_start_date", F.trunc(F.col("curve_start_month").try_cast("date"), "month"))
        .withColumn("target_is_imputed", F.col("actual_is_imputed").try_cast("boolean"))
        .where(F.col("game_key").isNotNull() & F.col("target_type").isNotNull()
               & (F.col("month_index") >= 1))
    )
    finite_actual = (F.col("raw_actual").isNotNull() & ~F.isnan("raw_actual")
                     & (F.abs(F.col("raw_actual")) != F.lit(float("inf"))))
    rows = rows.withColumn("trusted_actual",
                           F.when(finite_actual & (F.col("target_is_imputed") == F.lit(False)),
                                  F.col("raw_actual")))
    keys = ["game_key", "target_type"]
    month_keys = keys + ["month_index"]
    # Repeated actuals from similarity experiments are not extra transactions.
    conflicts = (rows.groupBy(*month_keys)
                 .agg(F.countDistinct("trusted_actual").alias("different_actuals"))
                 .where(F.col("different_actuals") > 1))
    start_conflicts = (rows.groupBy(*keys)
                       .agg(F.countDistinct("source_start_date").alias("different_starts"))
                       .where(F.col("different_starts") > 1))
    if conflicts.limit(1).count() or start_conflicts.limit(1).count():
        display(conflicts.limit(20))
        display(start_conflicts.limit(20))
        raise ValueError("Observed actuals or curve starts disagree across experiments. Select one source experiment.")

    monthly = (rows.groupBy(*month_keys)
               .agg(F.max("trusted_actual").alias("target_value"),
                    F.max("source_start_date").alias("source_start_date"),
                    F.max(F.when(finite_actual & F.col("target_is_imputed").isNull(), 1)
                          .otherwise(0)).alias("has_unknown_actual")))
    observed = F.col("target_value").isNotNull()
    coverage = (
        monthly.groupBy(*keys)
        .agg(F.max("source_start_date").alias("source_start_date"),
             F.coalesce(F.max(F.when(observed, F.col("month_index"))), F.lit(0)).alias("coverage_months"),
             F.min(F.when(observed & (F.col("month_index") >= horizon), F.col("month_index")))
              .alias("coverage_confirmation_index"),
             F.sum(F.when(observed & (F.col("month_index") <= horizon), 1).otherwise(0))
              .alias("observed_months_in_horizon"),
             F.max(F.when(F.col("month_index") <= horizon, F.col("has_unknown_actual"))
                   .otherwise(0)).alias("unknown_actual_in_horizon"))
        .withColumn("has_complete_horizon",
                    (F.col("coverage_months") >= horizon)
                    & F.col("source_start_date").isNotNull()
                    & (F.col("unknown_actual_in_horizon") == 0))
        .withColumn("target_window_end_month", F.add_months("source_start_date", horizon - 1))
        # A full month's outcome is available at the next month's start.
        # If month 8 proves months 5–7 were zero, the label is available in month 9.
        .withColumn("target_available_date", F.add_months("source_start_date", F.col("coverage_confirmation_index")))
        .withColumn("zero_filled_months", F.when(F.col("has_complete_horizon"),
                                                 F.lit(horizon) - F.col("observed_months_in_horizon")))
    )
    grid = (coverage.withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(horizon))))
            .join(monthly.select(*month_keys, F.col("target_value").alias("observed_actual")),
                  month_keys, "left"))
    by_month = Window.partitionBy(*keys).orderBy("month_index")
    sales_endpoint = F.coalesce(F.last("observed_actual", ignorenulls=True).over(
        by_month.rowsBetween(Window.unboundedPreceding, Window.currentRow)), F.lit(0.0))
    grid = grid.withColumn("filled_sales_cumulative", sales_endpoint)
    monthly_units = F.when(F.col("target_type") == "sales",
                           F.col("filled_sales_cumulative")
                           - F.coalesce(F.lag("filled_sales_cumulative").over(by_month), F.lit(0.0)))
    monthly_units = monthly_units.otherwise(F.coalesce(F.col("observed_actual"), F.lit(0.0)))
    zero_filled = (grid.withColumn("monthly_units", F.when(F.col("has_complete_horizon"), monthly_units))
                   .withColumn("is_zero_filled", F.col("has_complete_horizon") & F.col("observed_actual").isNull()))
    totals = zero_filled.groupBy(*keys).agg(F.sum("monthly_units").alias("target_value"))
    totals = (coverage.join(totals, keys, "left")
              .withColumn("target_is_imputed", F.when(F.col("has_complete_horizon"), F.lit(False)))
              .withColumn("target_horizon_months", F.lit(horizon)))
    return monthly, zero_filled, totals


monthly_targets_df, zero_filled_monthly_targets_df, six_month_targets_df = prepare_six_month_targets(
    df_demand_forecast, FORECAST_MONTHS)

source_start_dates_df = (six_month_targets_df.groupBy("game_key").agg(
    F.min(F.when(F.col("target_type") == "sales", F.col("source_start_date"))).alias("sales_start_date"),
    F.min(F.when(F.col("target_type") == "rental", F.col("source_start_date"))).alias("rental_start_date")))
ai_attr_clean_df = ai_attr_clean_df.join(source_start_dates_df, "game_key", "left")

# ============================================================
# 4. ONE row per AI game and revenue type, with static attributes
# ============================================================
model_input_df = (
    ai_attr_clean_df.withColumn("target_type", F.explode(F.array(F.lit("sales"), F.lit("rental"))))
    .join(six_month_targets_df, ["game_key", "target_type"], "left")
    .fillna({"has_complete_horizon": False, "coverage_months": 0})
)
require_unique(model_input_df, ["game_key", "target_type"], "Six-month model input")
usable_actual = F.col("has_complete_horizon") & F.col("target_value").isNotNull()
sales_training_df = model_input_df.where((F.col("target_type") == "sales") & usable_actual)
rental_training_df = model_input_df.where((F.col("target_type") == "rental") & usable_actual)

# No month_index predictor: each model learns a single six-month total.
feature_columns = numeric_features + categorical_features
unmatched_targets_df = six_month_targets_df.join(ai_attr_clean_df.select("game_key"), "game_key", "left_anti")
print(f"Numeric features: {len(numeric_features)}; categorical: {categorical_features}")
print("Sales = cumulative sales at month 6; rent = sum of monthly rentals 1–6 (unit-months).")
print("Eligibility: >=6 calendar months covered by actuals; interior gaps count as zero.")
print("Monthly audit: zero_filled_monthly_targets_df; one-row targets: six_month_targets_df.")
display(model_input_df.groupBy("target_type").agg(
    F.count("game_key").alias("all_games"),
    F.sum(F.when(usable_actual, 1).otherwise(0)).alias("games_with_complete_six_month_targets"),
    F.sum(F.when(~F.col("has_complete_horizon"), 1).otherwise(0)).alias("prediction_only_or_incomplete_games")))
display(unmatched_targets_df.limit(20))
display(six_month_targets_df.limit(20))


In [0]:
# ============================================================
# CELL 4 — Preprocessing, train/test split, tuning, and XGBoost models
# ============================================================
import uuid
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "Run %pip install xgboost in a separate cell. "
        "Restart Python if requested, then rerun cells 1–4."
    ) from exc


# ============================================================
# 1. Configuration
# ============================================================
EXPERIMENT_NAME = f"xgb_ai_attributes_total{FORECAST_MONTHS}m_zero_defaults_v6"
EXPERIMENT_ID = "xgb_" + uuid.uuid4().hex[:12]


# ============================================================
# 2. Convert the prepared modeling dataset to pandas
# ============================================================
metadata = [
    "game_key",
    "game_name",
    "target_type",
    "target_value",
    "target_is_imputed",
    "source_start_date",
    "target_window_end_month",
    "target_available_date",
    "game_release_date",
    "sales_start_date",
    "rental_start_date",
    "to_sales_name_match",
    "to_rental_name_match",
    "coverage_months",
    "coverage_confirmation_index",
    "observed_months_in_horizon",
    "zero_filled_months",
    "has_complete_horizon",
    "target_horizon_months",
]

model_pdf = model_input_df.select(
    *list(dict.fromkeys(metadata + feature_columns))
).toPandas()

for column in numeric_features + ["target_value", "coverage_months"]:
    model_pdf[column] = (
        pd.to_numeric(model_pdf[column], errors="coerce")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )

# Keep the semantic values visible as zero in saved inputs and SHAP tables.
for column in ZERO_DEFAULT_NUMERIC_FEATURES:
    if column in model_pdf:
        model_pdf[column] = model_pdf[column].fillna(0.0)

for column in categorical_features:
    model_pdf[column] = (
        model_pdf[column]
        .fillna("__missing__")
        .astype(str)
    )

date_columns = [
    "game_release_date",
    "sales_start_date",
    "rental_start_date",
    "source_start_date",
    "target_window_end_month",
    "target_available_date",
]

for column in date_columns:
    model_pdf[column] = pd.to_datetime(
        model_pdf[column],
        errors="coerce",
    )

# Only completed, confirmed six-month totals enter training/evaluation.
if model_pdf.duplicated(["game_key", "target_type"]).any():
    raise ValueError("Expected one row per game and target type.")
model_pdf["is_observed"] = (
    model_pdf["target_value"].notna()
    & model_pdf["has_complete_horizon"].eq(True).fillna(False)
    & model_pdf["coverage_months"].ge(FORECAST_MONTHS)
    & model_pdf["target_available_date"].notna()
    & model_pdf["target_is_imputed"].eq(False).fillna(False)
)


# ============================================================
# 3. Shared chronological split by GAME
# ============================================================
# Each game's sales and rental rows receive the same split.
# Each revenue type retains its own source start and six-month window.
game_dates = (
    model_pdf.groupby("game_key")["game_release_date"]
    .min()
)

commercial_dates = (
    model_pdf.loc[model_pdf["is_observed"]]
    .groupby("game_key")["source_start_date"]
    .min()
)

game_split_pdf = pd.concat(
    [
        game_dates.rename("metadata_release_date"),
        commercial_dates.rename("commercial_start_date"),
    ],
    axis=1,
)

game_split_pdf["first_observed_date"] = (
    game_split_pdf.min(axis=1)
)

labeled_games = set(
    model_pdf.loc[model_pdf["is_observed"], "game_key"]
)

eligible = (
    game_split_pdf.loc[
        game_split_pdf.index.isin(labeled_games)
        & game_split_pdf["first_observed_date"].notna()
    ]
    .sort_values("first_observed_date")
)

if len(eligible) < 10:
    raise ValueError(
        "Fewer than 10 dated games with complete six-month totals."
    )

test_count = max(
    1,
    int(np.ceil(TEST_FRACTION * len(eligible))),
)

TEST_CUTOFF = eligible.iloc[-test_count]["first_observed_date"]

game_split_pdf["split"] = "predict_only"

game_split_pdf.loc[
    game_split_pdf.index.isin(labeled_games),
    "split",
] = "undated"

game_split_pdf.loc[eligible.index, "split"] = np.where(
    eligible["first_observed_date"] < TEST_CUTOFF,
    "train",
    "test",
)

# Games tied on the cutoff date stay together.
# Therefore, the actual split may differ slightly from 80/20.
model_pdf = model_pdf.join(
    game_split_pdf[["first_observed_date", "split"]],
    on="game_key",
)


# ============================================================
# 4. Check for shared source curves crossing train/test
# ============================================================
for target_type, mapping in [
    ("sales", "to_sales_name_match"),
    ("rental", "to_rental_name_match"),
]:
    aliases = (
        model_pdf.loc[
            model_pdf["target_type"].eq(target_type)
            & model_pdf["is_observed"]
            & model_pdf["split"].isin(["train", "test"]),
            [mapping, "game_key", "split"],
        ]
        .drop_duplicates()
        .copy()
    )

    aliases[mapping] = (
        aliases[mapping]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    aliases = aliases.loc[
        aliases[mapping].notna()
        & ~aliases[mapping].isin(
            ["", "none", "null", "nan", "na", "n/a"]
        )
    ]

    overlap = aliases.groupby(mapping)["split"].nunique()
    overlap = overlap.index[overlap > 1]

    if len(overlap):
        display(
            aliases.loc[aliases[mapping].isin(overlap)]
        )
        raise ValueError(
            f"{target_type}: shared source curves cross train/test. "
            "Resolve these mappings or group the aliases before training."
        )


# ============================================================
# 5. Preprocessing helpers
# ============================================================
class MissingAsZero(BaseEstimator, TransformerMixin):
    """Preserve semantic absence as 0 in training and every later prediction."""
    def __init__(self, columns=()):
        self.columns = columns

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        result = X.copy()
        for column in self.columns:
            if column in result:
                result[column] = (pd.to_numeric(result[column], errors="coerce")
                                  .replace([np.inf, -np.inf], np.nan).fillna(0.0))
        return result

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_ if input_features is None else np.asarray(input_features)


class UpperQuantileCapper(BaseEstimator, TransformerMixin):
    """Learn upper caps for reels and paylines/ways."""

    def __init__(
        self,
        columns=("reels", "paylines_or_ways"),
        quantile=0.99,
    ):
        self.columns = columns
        self.quantile = quantile

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object,
        )

        self.caps_ = {
            column: float(X[column].quantile(self.quantile))
            for column in self.columns
            if column in X.columns
        }

        return self

    def transform(self, X):
        result = X.copy()

        for column, cap in self.caps_.items():
            if np.isfinite(cap):
                result[column] = result[column].clip(
                    upper=cap
                )

        return result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features)


def metric_record(frame, prediction_column):
    errors = frame[prediction_column].to_numpy() - frame["target_value"].to_numpy()
    denominator = float(frame["target_value"].abs().sum())
    return {
        "games": frame["game_key"].nunique(),
        "predictions": len(frame),
        "window": f"total_{FORECAST_MONTHS}_months",
        "MAE": float(np.mean(np.abs(errors))),
        "RMSE": float(np.sqrt(np.mean(errors ** 2))),
        "Bias": float(np.mean(errors)),
        "WAPE": float(np.abs(errors).sum() / denominator) if denominator > 0 else np.nan,
    }


# ============================================================
# Hyperparameter tuning inside the OUTER training split only
# ============================================================
from sklearn.model_selection import ParameterSampler
import json
import copy


def build_preprocessor(fit_features):
    num_cols = [c for c in numeric_features
                if c in ZERO_DEFAULT_NUMERIC_FEATURES or fit_features[c].notna().any()]
    dropped = sorted(set(numeric_features) - set(num_cols))
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("zero_defaults", MissingAsZero(columns=tuple(
                c for c in num_cols if c in ZERO_DEFAULT_NUMERIC_FEATURES))),
            ("cap", UpperQuantileCapper(quantile=CAP_QUANTILE)),
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", MinMaxScaler(clip=True)),
        ]), num_cols),
        ("category", encoder, categorical_features),
    ])
    preprocessor.fit(fit_features[feature_columns])
    return preprocessor, dropped, list(preprocessor.get_feature_names_out())


def equal_game_weights(frame):
    weights = 1.0 / frame.groupby("game_key")["game_key"].transform("size")
    return weights / weights.mean()


def make_tuning_folds(curve, training, target_type):
    # Source aliases stay together. The outer test games are absent from training.
    mapping = "to_sales_name_match" if target_type == "sales" else "to_rental_name_match"
    games = training[["game_key", "first_observed_date", mapping]].drop_duplicates("game_key").copy()
    source = (games[mapping].astype("string").str.strip().str.lower()
              .str.replace(r"\s+", " ", regex=True))
    source = source.mask(source.isin(["", "none", "null", "nan", "na", "n/a"]))
    games["source_group"] = ("source:" + source).fillna("game:" + games["game_key"].astype(str))
    games["cohort_date"] = games.groupby("source_group")["first_observed_date"].transform("min")
    dates = np.array(sorted(games["cohort_date"].dropna().unique()))
    warmup = max(1, int(np.ceil(len(dates) * TUNING_INITIAL_TRAIN_FRACTION)))
    date_blocks = np.array_split(dates[warmup:], TUNING_FOLDS)
    folds, audit, memberships = [], [], []
    for fold_number, dates_block in enumerate(date_blocks, 1):
        if len(dates_block) == 0:
            continue
        cutoff = pd.Timestamp(dates_block[0])
        fit_keys = set(games.loc[games["cohort_date"].lt(cutoff)
                                 & games["first_observed_date"].lt(cutoff), "game_key"])
        valid_keys = set(games.loc[games["cohort_date"].isin(dates_block), "game_key"])
        # A total can enter fit only once its full horizon and coverage proof are known.
        fit = training.loc[training["game_key"].isin(fit_keys)
                           & training["target_available_date"].le(cutoff)].copy()
        valid = training.loc[training["game_key"].isin(valid_keys)
                             & training["target_available_date"].le(TEST_CUTOFF)].copy()
        fit_sources = set(games.loc[games["game_key"].isin(fit["game_key"]), "source_group"])
        valid_sources = set(games.loc[games["game_key"].isin(valid["game_key"]), "source_group"])
        usable = (len(fit_sources) >= TUNING_MIN_TRAIN_GROUPS
                  and len(valid_sources) >= TUNING_MIN_VALIDATION_GROUPS)
        record = {
            "target_type": target_type, "fold": fold_number, "cutoff": cutoff,
            "last_validation_cohort_date": pd.Timestamp(dates_block[-1]),
            "training_games": fit["game_key"].nunique(), "validation_games": valid["game_key"].nunique(),
            "training_source_groups": len(fit_sources), "validation_source_groups": len(valid_sources),
            "training_rows": len(fit), "validation_rows": len(valid),
            "latest_training_label_available_date": fit["target_available_date"].max(),
            "latest_validation_label_available_date": valid["target_available_date"].max(),
            "status": "used" if usable else "insufficient_history",
        }
        audit.append(record)
        if not usable:
            continue
        assert not fit_sources & valid_sources
        assert not set(fit["game_key"]) & set(valid["game_key"])
        assert fit["split"].eq("train").all() and valid["split"].eq("train").all()
        assert fit["target_available_date"].le(cutoff).all()
        assert valid["target_available_date"].le(TEST_CUTOFF).all()
        preprocessor, _, _ = build_preprocessor(fit)
        folds.append({
            "fold": fold_number, "fit_x": preprocessor.transform(fit[feature_columns]),
            "fit_y": fit["target_value"].to_numpy(), "fit_weights": equal_game_weights(fit).to_numpy(),
            "valid_x": preprocessor.transform(valid[feature_columns]),
            "valid_y": valid["target_value"].to_numpy(), "valid_games": valid["game_key"].to_numpy(),
            "clip_at_zero": bool(fit["target_value"].ge(0).all()),
        })
        for role, sample in [("train", fit), ("validation", valid)]:
            members = sample.groupby("game_key").agg(
                observed_rows=("game_key", "size"),
                target_window_end_month=("target_window_end_month", "max"),
                target_available_date=("target_available_date", "max"),
            ).reset_index().merge(games[["game_key", "source_group", "cohort_date"]], on="game_key", validate="one_to_one")
            members["target_type"] = target_type
            members["fold"] = fold_number
            members["fold_cutoff"] = cutoff
            members["role"] = role
            memberships.extend(members.to_dict("records"))
    return folds, audit, memberships


def tune_model(curve, training, target_type):
    base = dict(XGB_PARAMS)
    summary = {"target_type": target_type, "status": "disabled", "selected_trial": None,
               "folds": 0, "trials": 0, "metric": TUNING_METRIC,
               "default_validation_score": None, "best_validation_score": None,
               "validation_improvement_pct": None}
    if not TUNE_HYPERPARAMETERS:
        return base, summary
    folds, audit, membership = make_tuning_folds(curve, training, target_type)
    tuning_fold_rows.extend(audit)
    tuning_membership_rows.extend(membership)
    summary["folds"] = len(folds)
    if len(folds) < TUNING_MIN_FOLDS:
        summary["status"] = "insufficient_validation_history"
        print(f"{target_type}: only {len(folds)} usable chronological folds; retaining configured parameters.")
        return base, summary

    candidates = [base]
    seen = {json.dumps(base, sort_keys=True)}
    if TUNING_TRIALS > 1:
        for sampled in ParameterSampler(TUNING_SEARCH_SPACE, n_iter=TUNING_TRIALS - 1, random_state=RANDOM_SEED):
            params = {**base, **sampled}
            # Convert numpy scalar values to JSON-native values for durable logging.
            params = {k: v.item() if isinstance(v, np.generic) else v for k, v in params.items()}
            signature = json.dumps(params, sort_keys=True)
            if signature not in seen:
                seen.add(signature)
                candidates.append(params)
    scores = []
    for trial, params in enumerate(candidates):
        fold_scores = []
        for fold in folds:
            model = XGBRegressor(**params)
            model.fit(fold["fit_x"], fold["fit_y"], sample_weight=fold["fit_weights"])
            predictions = np.asarray(model.predict(fold["valid_x"]), dtype=float)
            if fold["clip_at_zero"]:
                predictions = np.maximum(predictions, 0)
            if not np.isfinite(predictions).all():
                raise ValueError(f"Non-finite tuning predictions: {target_type}, trial {trial}, fold {fold['fold']}.")
            error = predictions - fold["valid_y"]
            per_game = pd.DataFrame({"game_key": fold["valid_games"], "abs_error": np.abs(error),
                                     "squared_error": error ** 2, "error": error}).groupby("game_key").mean()
            metrics = {
                "target_type": target_type, "trial": trial, "fold": fold["fold"],
                "validation_games": len(per_game), "validation_rows": len(error),
                "game_weighted_mae": float(per_game["abs_error"].mean()),
                "game_weighted_mse": float(per_game["squared_error"].mean()),
                "game_weighted_bias": float(per_game["error"].mean()),
                "row_mae": float(np.mean(np.abs(error))), "row_mse": float(np.mean(error ** 2)),
            }
            fold_scores.append(metrics)
            tuning_trial_fold_rows.append(metrics)
        # Validation cohorts are disjoint. Weight folds by games to give each
        # validation game equal influence.
        game_weights = np.array([r["validation_games"] for r in fold_scores])
        row_weights = np.array([r["validation_rows"] for r in fold_scores])
        weighted = lambda field, weights: float(np.average([r[field] for r in fold_scores], weights=weights))
        row = {
            "target_type": target_type, "trial": trial, "is_configured_default": trial == 0,
            "folds": len(folds), "validation_games": int(game_weights.sum()),
            "validation_rows": int(row_weights.sum()),
            "game_weighted_mae": weighted("game_weighted_mae", game_weights),
            "game_weighted_rmse": float(np.sqrt(weighted("game_weighted_mse", game_weights))),
            "game_weighted_bias": weighted("game_weighted_bias", game_weights),
            "row_mae": weighted("row_mae", row_weights),
            "row_rmse": float(np.sqrt(weighted("row_mse", row_weights))),
            "params_json": json.dumps(params, sort_keys=True),
        }
        row["selection_score"] = row["game_weighted_" + TUNING_METRIC.lower()]
        scores.append(row)
        print(f"{target_type} trial {trial + 1}/{len(candidates)}: validation {TUNING_METRIC} = {row['selection_score']:.3f}")
    # The configured default wins an exact tie, so complexity does not win by accident.
    best = min(scores, key=lambda row: (row["selection_score"], row["trial"]))
    for row in scores:
        row["selected"] = row["trial"] == best["trial"]
        tuning_trial_rows.append(row)
    default_score = scores[0]["selection_score"]
    summary.update(status="tuned", selected_trial=best["trial"], trials=len(scores),
                   default_validation_score=default_score, best_validation_score=best["selection_score"],
                   validation_improvement_pct=(100 * (default_score - best["selection_score"]) / default_score
                                               if default_score > 0 else None))
    return candidates[best["trial"]], summary


# ============================================================
# 6. Train the two models
# ============================================================
xgb_models = {}
xgb_model_info = {}
best_params_by_target = {}
tuning_trial_rows, tuning_trial_fold_rows = [], []
tuning_fold_rows, tuning_membership_rows, tuning_summary_rows = [], [], []

prediction_frames = []
metric_rows = []
game_metric_rows = []
split_rows = []

for target_type in ["sales", "rental"]:
    curve = (
        model_pdf.loc[
            model_pdf["target_type"].eq(target_type)
        ]
        .copy()
        .sort_values("game_key")
    )

    # Completed labels, including later gap-confirmation records,
    # must be available by the historical test cutoff.
    train_mask = (
        curve["split"].eq("train")
        & curve["is_observed"]
        & curve["target_available_date"].le(TEST_CUTOFF)
    )

    test_mask = (
        curve["split"].eq("test")
        & curve["is_observed"]
    )

    training = curve.loc[train_mask]
    test = curve.loc[test_mask]

    n_train = training["game_key"].nunique()
    n_test = test["game_key"].nunique()

    if n_train < 5 or n_test < 1:
        print(
            f"Skipping {target_type}: "
            f"{n_train} training / {n_test} test games."
        )
        continue

    # One row per game: fit transforms only on eligible training totals.
    fit_features = training

    # Select parameters using only chronological folds inside the outer train split.
    selected_params, tuning_summary = tune_model(curve, training, target_type)
    best_params_by_target[target_type] = dict(selected_params)
    tuning_summary_rows.append(tuning_summary)

    # Refit preprocessing and the winning configuration on ALL outer training rows.
    preprocessing, dropped, output_features = build_preprocessor(fit_features)
    train_x = preprocessing.transform(training[feature_columns])
    weights = equal_game_weights(training)
    model = XGBRegressor(**selected_params)

    model.fit(
        train_x,
        training["target_value"],
        sample_weight=weights,
    )

    fitted_pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessing),
            ("model", model),
        ]
    )

    xgb_models[target_type] = fitted_pipeline

    # Predict ONE total for every AI game, including games without eligible actuals.
    curve["raw_prediction"] = fitted_pipeline.predict(curve[feature_columns])
    curve["prediction"] = curve["raw_prediction"]

    clip_at_zero = bool(
        training["target_value"].ge(0).all()
    )

    if clip_at_zero:
        curve["prediction"] = curve["prediction"].clip(
            lower=0
        )

    # One training-mean TOTAL baseline per revenue type.
    curve["baseline_prediction"] = float(training["target_value"].mean())

    curve["used_for_fit"] = train_mask
    curve["used_for_test"] = test_mask
    curve["experiment_id"] = EXPERIMENT_ID
    curve["experiment_name"] = EXPERIMENT_NAME

    prediction_frames.append(curve)

    # --------------------------------------------------------
    # Evaluate only observed test targets
    # --------------------------------------------------------
    evaluation = curve.loc[test_mask]

    for estimator, prediction_column in [
        ("xgboost", "prediction"),
        ("training_mean", "baseline_prediction"),
    ]:
        metric_rows.append(
            {
                "target_type": target_type,
                "estimator": estimator,
                **metric_record(
                    evaluation,
                    prediction_column,
                ),
            }
        )

    for game_key, group in evaluation.groupby("game_key"):
        game_metric_rows.append(
            {
                "target_type": target_type,
                "game_key": game_key,
                **metric_record(group, "prediction"),
            }
        )

    # Retain preprocessing/model information for later prediction.
    xgb_model_info[target_type] = {
        "experiment_id": EXPERIMENT_ID,
        "experiment_name": EXPERIMENT_NAME,
        "test_cutoff": TEST_CUTOFF,
        "feature_columns": list(feature_columns),
        "zero_default_numeric_features": list(ZERO_DEFAULT_NUMERIC_FEATURES),
        "dropped_empty_features": dropped,
        "transformed_features": output_features,
        "prediction_kind": "six_month_total",
        "horizon_months": FORECAST_MONTHS,
        "clip_predictions_at_zero": clip_at_zero,
        "selected_params": dict(selected_params),
        "tuning": dict(tuning_summary),
        "upper_caps": getattr(preprocessing.named_transformers_["numeric"].named_steps["cap"], "caps_", {}),
    }

    split_rows.append(
        {
            "target_type": target_type,
            "training_games": n_train,
            "test_games": n_test,
            "training_totals": len(training),
            "test_totals": len(test),
            "full_horizon_test_games": len(test),
            "zero_filled_training_months": int(training["zero_filled_months"].sum()),
            "zero_filled_test_months": int(test["zero_filled_months"].sum()),
            "unavailable_training_totals_excluded": int((
                curve["split"].eq("train") & curve["is_observed"]
                & curve["target_available_date"].gt(TEST_CUTOFF)
            ).sum()),
        }
    )

    print(
        f"{target_type}: fitted on {n_train} games; "
        f"test on {n_test} games."
    )
    print(f"Dropped all-missing numeric features: {dropped}")


# ============================================================
# 7. Results
# ============================================================
if not prediction_frames:
    raise ValueError(
        "Neither model had enough dated training/test observations."
    )

xgb_predictions_pdf = pd.concat(
    prediction_frames,
    ignore_index=True,
)

xgb_metrics_pdf = pd.DataFrame(metric_rows)
xgb_game_metrics_pdf = pd.DataFrame(game_metric_rows)
xgb_split_summary_pdf = pd.DataFrame(split_rows)
tuning_summary_pdf = pd.DataFrame(tuning_summary_rows)
tuning_trials_pdf = pd.DataFrame(tuning_trial_rows)
tuning_trial_folds_pdf = pd.DataFrame(tuning_trial_fold_rows)
tuning_folds_pdf = pd.DataFrame(tuning_fold_rows)
tuning_membership_pdf = pd.DataFrame(tuning_membership_rows)
for frame in [tuning_summary_pdf, tuning_trials_pdf, tuning_trial_folds_pdf,
              tuning_folds_pdf, tuning_membership_pdf]:
    frame["experiment_id"] = EXPERIMENT_ID
    frame["experiment_name"] = EXPERIMENT_NAME
print("Tuning status and validation improvement over the configured default:")
display(tuning_summary_pdf)
if not tuning_trials_pdf.empty:
    display(tuning_trials_pdf.sort_values(["target_type", "selection_score", "trial"]))
if not tuning_folds_pdf.empty:
    display(tuning_folds_pdf)

for result in [xgb_metrics_pdf, xgb_game_metrics_pdf]:
    result["experiment_id"] = EXPERIMENT_ID
    result["experiment_name"] = EXPERIMENT_NAME

print(f"\nExperiment: {EXPERIMENT_ID}")
print(f"Shared test cutoff: {TEST_CUTOFF.date()}")
print("Sales target: total units across source calendar months 1–6.")
print("Rental target: sum of monthly units across source calendar months 1–6 (unit-months).")
print("Bias = prediction - actual; positive means overprediction.")
print("Metrics give each eligible six-month test total equal weight.")
print("Models are available in xgb_models.")
print("One total prediction per game/type is in xgb_predictions_pdf.")

display(xgb_split_summary_pdf)

display(xgb_metrics_pdf)

display(
    xgb_game_metrics_pdf.sort_values(
        ["target_type", "RMSE"],
        ascending=[True, False],
    )
)

# Capture the configuration used by these fitted models for MLflow provenance.
BASELINE_TRAINING_CONFIG = {
    "FORECAST_MONTHS": FORECAST_MONTHS, "TEST_FRACTION": TEST_FRACTION,
    "CAP_QUANTILE": CAP_QUANTILE, "RANDOM_SEED": RANDOM_SEED,
    "ZERO_DEFAULT_NUMERIC_FEATURES": list(ZERO_DEFAULT_NUMERIC_FEATURES),
    "XGB_PARAMS": copy.deepcopy(XGB_PARAMS), "EXPERIMENT_ID": EXPERIMENT_ID,
    **{name: copy.deepcopy(globals()[name]) for name in [
        "TUNE_HYPERPARAMETERS", "TUNING_TRIALS", "TUNING_FOLDS", "TUNING_MIN_FOLDS",
        "TUNING_INITIAL_TRAIN_FRACTION", "TUNING_MIN_TRAIN_GROUPS",
        "TUNING_MIN_VALIDATION_GROUPS", "TUNING_METRIC", "TUNING_SEARCH_SPACE",
    ]},
}



In [0]:
%pip install xgboost

In [0]:
# ============================================================
# CELL 5 — Select a game and explain its single six-month total
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

from xgboost import DMatrix

TEST_ONLY = True
TOP_FEATURES = 15

# The old monthly explanation selector no longer applies to a scalar target.
try:
    dbutils.widgets.remove("xgb_shap_month")
except Exception:
    pass

eligible = xgb_predictions_pdf.loc[
    xgb_predictions_pdf["target_type"].isin(xgb_models)
].copy()
if TEST_ONLY:
    eligible = eligible.loc[
        eligible["split"].eq("test")
        & eligible["used_for_test"].eq(True).fillna(False)
    ]

if eligible.duplicated(["game_key", "target_type"]).any():
    raise ValueError("Expected one six-month total per game/model. Rerun cell 4.")

game_list = (
    eligible[["game_key", "target_type", "split"]]
    .drop_duplicates()
    .sort_values(["game_key", "target_type"])
)
if game_list.empty:
    raise ValueError("No selectable games. Run cell 4 or set TEST_ONLY = False.")

labels = {"sales": "Sale", "rental": "Rent"}
game_options = {
    f"{r.game_key} | {labels[r.target_type]} | {r.split}": (r.game_key, r.target_type)
    for r in game_list.itertuples(index=False)
}


def dropdown_value(name, choices, label):
    default = choices[0]
    try:
        previous = dbutils.widgets.get(name)
        if previous in choices:
            default = previous
    except Exception:
        pass
    dbutils.widgets.dropdown(name, default, choices, label)
    selected = dbutils.widgets.get(name)
    if selected not in choices:
        raise ValueError(f"Choose a current value for {label}, then rerun this cell.")
    return selected


selected_label = dropdown_value("xgb_game_shap", list(game_options), "Game / model")
selected_game, target_type = game_options[selected_label]
rows = eligible.loc[
    eligible["game_key"].eq(selected_game)
    & eligible["target_type"].eq(target_type)
].reset_index(drop=True)
if len(rows) != 1:
    raise ValueError("Expected exactly one total prediction for this game/model.")
row = rows.iloc[0]

# Apply only the fitted training transforms. This cell never refits a model.
pipeline = xgb_models[target_type]
info = xgb_model_info[target_type]
preprocessor = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]
booster = model.get_booster()
X = preprocessor.transform(rows[info["feature_columns"]])
feature_names = list(preprocessor.get_feature_names_out())
dmatrix = DMatrix(
    X,
    feature_names=booster.feature_names,
    missing=model.get_params().get("missing", np.nan),
)
best_iteration = getattr(model, "best_iteration", None)
tree_range = {} if best_iteration is None else {"iteration_range": (0, best_iteration + 1)}

# Native TreeSHAP for this ONE total. The last column is the SHAP base value.
contributions = np.asarray(booster.predict(
    dmatrix, pred_contribs=True, approx_contribs=False, **tree_range,
))
if contributions.ndim == 3 and contributions.shape[1] == 1:
    contributions = contributions[:, 0, :]
if contributions.shape != (1, len(feature_names) + 1):
    raise ValueError("Unexpected SHAP shape. Expected single-output total regression.")

shap_values = contributions[0, :-1]
base_value = float(contributions[0, -1])
raw_prediction = float(np.asarray(model.predict(X)).reshape(-1)[0])
reconstructed = base_value + float(shap_values.sum())
tolerance = 1e-4 * max(1.0, float(np.abs(contributions).sum()))
if abs(reconstructed - raw_prediction) > tolerance:
    raise ValueError("SHAP values do not reconstruct this model's raw total prediction.")

clip_at_zero = info.get("clip_predictions_at_zero", info.get("clip_at_zero", False))
prediction = max(raw_prediction, 0.0) if clip_at_zero else raw_prediction
if not np.isclose(prediction, float(row["prediction"]), rtol=1e-5, atol=1e-4):
    raise ValueError("This model and stored prediction disagree. Rerun cells 4 and 5.")

feature_values = X.toarray() if hasattr(X, "toarray") else np.asarray(X)


def source_feature(transformed_name):
    """Associate encoded/scaled values and missing indicators with raw inputs."""
    name = transformed_name.split("__", 1)[-1]
    if name.startswith("missingindicator_"):
        name = name.removeprefix("missingindicator_")
    for original in sorted(info["feature_columns"], key=len, reverse=True):
        if name == original or (
            transformed_name.startswith("category__") and name.startswith(original + "_")
        ):
            return original
    return None


selected_total_shap_pdf = pd.DataFrame({
    "feature": feature_names,
    "model_feature_value": feature_values.reshape(-1),
    "shap_units": shap_values,
})
selected_total_shap_pdf["source_feature"] = selected_total_shap_pdf["feature"].map(source_feature)
selected_total_shap_pdf["original_feature_value"] = selected_total_shap_pdf["source_feature"].map(
    lambda name: None if name is None or pd.isna(row[name]) else str(row[name])
)
selected_total_shap_pdf["abs_shap"] = selected_total_shap_pdf["shap_units"].abs()
selected_total_shap_pdf = selected_total_shap_pdf.sort_values(
    "abs_shap", ascending=False,
).reset_index(drop=True)

# A wide version is also available for downstream comparisons across games.
selected_game_shap_pdf = pd.DataFrame([dict(zip(feature_names, shap_values))])
selected_game_shap_pdf.insert(0, "target_type", target_type)
selected_game_shap_pdf.insert(0, "game_key", selected_game)
selected_game_shap_pdf["forecast_months"] = FORECAST_MONTHS
selected_game_shap_pdf["base_value"] = base_value
selected_game_shap_pdf["raw_prediction"] = raw_prediction
selected_game_shap_pdf["prediction"] = prediction

# Display the leading contributions and preserve the sum of all remaining ones.
top = selected_total_shap_pdf.head(TOP_FEATURES)[["feature", "shap_units"]].copy()
remaining = selected_total_shap_pdf.iloc[TOP_FEATURES:]
if not remaining.empty:
    top.loc[len(top)] = ["Other features (sum)", remaining["shap_units"].sum()]


def readable_feature(name):
    name = name.removeprefix("numeric__").removeprefix("category__")
    return textwrap.fill(name.replace("_", " "), width=34)


actual_available = bool(row["is_observed"]) and pd.notna(row["target_value"])
actual_total = float(row["target_value"]) if actual_available else np.nan
baseline_total = float(row["baseline_prediction"])
units = "Sales units" if target_type == "sales" else "Rental unit-months"

fig, (ax_total, ax_shap) = plt.subplots(
    1, 2, figsize=(19, 9), gridspec_kw={"width_ratios": [1, 1.2]},
    constrained_layout=True,
)
bar_names, bar_values, bar_colors = ["XGBoost"], [prediction], ["#006BB6"]
if actual_available:
    bar_names.append("Actual total")
    bar_values.append(actual_total)
    bar_colors.append("#E57A21")
if np.isfinite(baseline_total):
    bar_names.append("Training mean")
    bar_values.append(baseline_total)
    bar_colors.append("#888888")
bar_rects = ax_total.bar(bar_names, bar_values, color=bar_colors, width=0.60)
for rect, value in zip(bar_rects, bar_values):
    ax_total.annotate(
        f"{value:,.1f}", (rect.get_x() + rect.get_width() / 2, value),
        ha="center", va="bottom" if value >= 0 else "top",
        xytext=(0, 5 if value >= 0 else -5), textcoords="offset points",
    )
ax_total.set(
    title=f"{selected_game}\n{labels[target_type]} | {row['split']} | First {FORECAST_MONTHS} months total",
    ylabel=units,
)
ax_total.margins(y=0.15)
ax_total.grid(axis="y", alpha=0.2)
ax_total.set_axisbelow(True)
if not actual_available:
    ax_total.text(0.02, 0.98, "Complete actual total unavailable", transform=ax_total.transAxes, va="top")

y = np.arange(len(top))
ax_shap.barh(
    y, top["shap_units"],
    color=np.where(top["shap_units"] >= 0, "#006BB6", "#E57A21"),
)
ax_shap.set_yticks(y)
ax_shap.set_yticklabels([readable_feature(name) for name in top["feature"]], fontsize=9)
ax_shap.invert_yaxis()
ax_shap.axvline(0, color="#444444", linewidth=0.8)
ax_shap.set(
    title=f"SHAP: contribution to the {FORECAST_MONTHS}-month total\nBase = {base_value:,.1f}; raw prediction = {raw_prediction:,.1f}",
    xlabel=f"Contribution in {units.lower()} (+ increases; − decreases)",
)
ax_shap.grid(axis="x", alpha=0.2)
plt.show()

summary = {
    "game_key": selected_game, "target_type": target_type,
    "forecast_months": FORECAST_MONTHS, "prediction": prediction,
    "actual_total": actual_total, "baseline_total": baseline_total,
}
for name in [
    "source_start_date", "target_window_end_month", "coverage_months", "has_complete_horizon",
    "actual_bracket", "predicted_bracket", "bracket_correct",
]:
    if name in row.index:
        summary[name] = row[name]
selected_total_summary_pdf = pd.DataFrame([summary])
display(selected_total_summary_pdf)
print(
    f"First {FORECAST_MONTHS} months: SHAP base {base_value:,.1f} "
    f"+ feature contributions {shap_values.sum():+,.1f} = raw total {raw_prediction:,.1f}."
)
if not np.isclose(raw_prediction, prediction):
    print(f"A zero floor adds {prediction - raw_prediction:,.1f}; displayed total = {prediction:,.1f}.")
if target_type == "rental":
    print("Rental total sums monthly rental units across six calendar months (unit-months).")
print("Positive SHAP values raise the total; negative values lower it. These are model contributions, not causal effects.")
print("original_feature_value is the raw input; model_feature_value is the encoded/scaled input used by XGBoost.")
display(selected_total_shap_pdf.drop(columns="abs_shap"))


In [0]:
# CELL 6 — MLflow tracking for the fitted six-month total sales and rent models
# Run cells 1–4 first. Cell 5 is an optional forecast/SHAP viewer.
# Each execution logs parent -> final target models -> validation trial runs.
# Trial results are recorded here after cell 4 finishes tuning and refitting.
import inspect
import json
import numpy as np
import pandas as pd
import tempfile
from pathlib import Path
from importlib.metadata import version
from sklearn.base import BaseEstimator, RegressorMixin

try:
    import mlflow
    import mlflow.sklearn
    import mlflow.pyfunc
    from mlflow.models import infer_signature
except ImportError as exc:
    raise ImportError("Install mlflow in Databricks, then rerun this cell.") from exc


class ForecastModel(RegressorMixin, BaseEstimator):
    """One cleaned game-attribute row -> one complete six-month demand total."""
    def __init__(self, pipeline, numeric, categorical, horizon, clip_at_zero):
        self.pipeline = pipeline
        self.numeric = numeric
        self.categorical = categorical
        self.horizon = horizon
        self.clip_at_zero = clip_at_zero

    def __sklearn_is_fitted__(self):
        return True

    def predict(self, X):
        data = pd.DataFrame(X).copy()
        columns = self.numeric + self.categorical
        missing = set(columns) - set(data.columns)
        if missing:
            raise ValueError(f"Missing cleaned input columns: {sorted(missing)}")
        for column in self.numeric:
            data[column] = pd.to_numeric(data[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        for column in self.categorical:
            data[column] = data[column].fillna("__missing__").astype(str)
        if data.empty:
            return np.array([], dtype=float)
        result = self.pipeline.predict(data[columns])
        return np.maximum(result, 0) if self.clip_at_zero else result


def log_frame(frame, name):
    with tempfile.TemporaryDirectory(prefix="demand_mlflow_") as directory:
        path = Path(directory) / f"{name}.csv.gz"
        frame.to_csv(path, index=False, compression="gzip")
        mlflow.log_artifact(str(path), artifact_path="tables")


def log_scores(frame, prefix):
    for row in frame.to_dict("records"):
        window = row.get("window", f"total_{FORECAST_MONTHS}_months")
        metrics = {f"{prefix}/{window}/{name}": float(row[name])
                   for name in ["MAE", "RMSE", "Bias", "WAPE", "games", "predictions"]
                   if name in row and pd.notna(row[name]) and np.isfinite(row[name])}
        if metrics:
            mlflow.log_metrics(metrics)


if "BASELINE_TRAINING_CONFIG" not in globals():
    raise ValueError("Run cells 1–4 before logging to MLflow.")
for name, fitted_value in BASELINE_TRAINING_CONFIG.items():
    if name in globals() and globals()[name] != fitted_value:
        raise ValueError(f"{name} changed after training. Rerun cells 1–4 before logging.")

experiment_path = MLFLOW_EXPERIMENT_PATH
if not experiment_path:
    current_user = spark.sql("SELECT current_user() AS user_name").first()["user_name"]
    experiment_path = f"/Users/{current_user}/AI_feature_extraction_forecasts"
if not experiment_path.startswith("/"):
    raise ValueError("MLFLOW_EXPERIMENT_PATH must be an absolute Databricks experiment path.")
experiment = mlflow.set_experiment(experiment_path)
MLFLOW_EXPERIMENT_ID = experiment.experiment_id
requirements = [f"mlflow=={mlflow.__version__}"] + [
    f"{package}=={version(package)}"
    for package in ["cloudpickle", "numpy", "pandas", "scipy", "scikit-learn", "xgboost"]
]
mlflow_forecast_models, run_records, tuning_run_records = {}, [], []

with mlflow.start_run(
    experiment_id=MLFLOW_EXPERIMENT_ID,
    run_name=MLFLOW_RUN_NAME or EXPERIMENT_NAME,
    nested=mlflow.active_run() is not None,
    tags={"project": "AI feature extraction", "forecast_origin": "prelaunch",
          "model_family": "attributes_only", "prediction_kind": "six_month_total",
          "experiment_id": EXPERIMENT_ID},
) as parent:
    MLFLOW_PARENT_RUN_ID = parent.info.run_id
    mlflow.log_params({
        "forecast_months": FORECAST_MONTHS, "test_fraction": TEST_FRACTION,
        "test_cutoff": str(TEST_CUTOFF.date()), "random_seed": RANDOM_SEED,
        "cap_quantile": CAP_QUANTILE,
        "tuning_enabled": TUNE_HYPERPARAMETERS, "tuning_trials_requested": TUNING_TRIALS,
        "tuning_folds_requested": TUNING_FOLDS, "tuning_metric": TUNING_METRIC,
        "tuning_weighting": "equal_weight_per_validation_game",
        "prediction_kind": "six_month_total",
        "sales_target": "first_6_calendar_months_total_units",
        "rental_target": "first_6_calendar_months_sum_of_monthly_units",
        "target_eligibility": "confirmed_source_coverage_at_least_6_months",
        "missing_months": "zero_only_within_confirmed_source_coverage",
        "training_label_rule": "target_available_date_le_test_cutoff",
        **{"base_xgb/" + k: v for k, v in BASELINE_TRAINING_CONFIG["XGB_PARAMS"].items()},
    })
    mlflow.log_dict({
        "training_config": BASELINE_TRAINING_CONFIG,
        "source_tables": SOURCE_TABLES,
        "selected_parameters": best_params_by_target,
        "tuning_method": "Random search with default candidate; expanding chronological game/source-group folds inside training only.",
        "numeric_features": numeric_features, "categorical_features": categorical_features,
        "requirements": requirements,
        "missing_attribute_policy": {
            "zero_default_numeric_features": list(ZERO_DEFAULT_NUMERIC_FEATURES),
            "meaning": "Missing paylines/ways, persistence, themes and mechanics mean 0; explicit zero paylines are valid.",
            "application": "Applied to cleaned source values and within the fitted pipeline before capping/imputation/scaling; reused by exported inference and SHAP.",
        },
        "data_snapshot_logged": MLFLOW_LOG_DATA_SNAPSHOT,
        "evaluation": "One complete six-month total per game/revenue type; shared chronological game split; training labels must be available by the historical cutoff.",
        "target_definitions": {
            "sales": "Cumulative sales at the end of source calendar month 6, forward-filled across confirmed zero-sale gaps. Cumulative rows are never summed.",
            "rental": "Sum monthly rental units over source calendar months 1 through 6. Units are rental unit-months, not distinct rented cabinets.",
            "source_start": "Sales and rentals retain their respective source start months.",
            "eligibility": "Require confirmed non-imputed source coverage through month 6; fewer than six nonzero months can still qualify.",
            "zero_gaps": "An absent month means zero only when a later observed record confirms source coverage; trailing unconfirmed months do not create a label.",
            "label_availability": "target_available_date is when the completed horizon and any zero gaps are supported by observed records. A later coverage-proof record can delay availability beyond the horizon end.",
        },
    }, "configuration.json")

    for target_type, pipeline in xgb_models.items():
        with mlflow.start_run(
            experiment_id=MLFLOW_EXPERIMENT_ID, run_name=f"xgb_{target_type}", nested=True,
            tags={"target_type": target_type, "model_family": "attributes_only",
                  "prediction_kind": "six_month_total"},
        ) as child:
            info = xgb_model_info[target_type]
            summary = info["tuning"]
            mlflow.log_params({"tuning/status": summary["status"],
                               "tuning/trials": summary["trials"],
                               "tuning/folds": summary["folds"],
                               "tuning/metric": summary["metric"],
                               "tuning/selected_trial": str(summary["selected_trial"])})
            mlflow.log_dict(info["selected_params"], "selected_parameters.json")
            validation_scores = {f"validation/{key}": float(summary[key])
                                 for key in ["default_validation_score", "best_validation_score",
                                             "validation_improvement_pct"]
                                 if summary[key] is not None and np.isfinite(summary[key])}
            if validation_scores:
                mlflow.log_metrics(validation_scores)

            if not tuning_trials_pdf.empty:
                trials = tuning_trials_pdf.loc[tuning_trials_pdf["target_type"].eq(target_type)]
                for trial in trials.to_dict("records"):
                    with mlflow.start_run(
                        experiment_id=MLFLOW_EXPERIMENT_ID,
                        run_name=f"{target_type}_trial_{int(trial['trial']):03d}", nested=True,
                        tags={"target_type": target_type, "run_role": "tuning_trial",
                              "selected": str(bool(trial["selected"])).lower(),
                              "experiment_id": EXPERIMENT_ID},
                    ) as trial_run:
                        mlflow.log_params(json.loads(trial["params_json"]))
                        mlflow.log_params({"selection_metric": TUNING_METRIC,
                                           "is_configured_default": trial["is_configured_default"],
                                           "folds": trial["folds"],
                                           "validation_games": trial["validation_games"],
                                           "validation_rows": trial["validation_rows"]})
                        mlflow.log_metrics({f"validation/{name}": float(trial[name]) for name in [
                            "selection_score", "game_weighted_mae", "game_weighted_rmse",
                            "game_weighted_bias", "row_mae", "row_rmse",
                        ]})
                        trial_folds = tuning_trial_folds_pdf.loc[
                            tuning_trial_folds_pdf["target_type"].eq(target_type)
                            & tuning_trial_folds_pdf["trial"].eq(trial["trial"])]
                        for fold in trial_folds.to_dict("records"):
                            mlflow.log_metrics({
                                f"fold_{int(fold['fold'])}/game_weighted_RMSE": float(np.sqrt(fold["game_weighted_mse"])),
                                f"fold_{int(fold['fold'])}/game_weighted_MAE": float(fold["game_weighted_mae"]),
                            })
                        tuning_run_records.append({
                            "target_type": target_type, "trial": int(trial["trial"]),
                            "selected": bool(trial["selected"]),
                            "mlflow_trial_run_id": trial_run.info.run_id,
                            "mlflow_model_run_id": child.info.run_id,
                            "mlflow_parent_run_id": MLFLOW_PARENT_RUN_ID,
                            "mlflow_experiment_id": MLFLOW_EXPERIMENT_ID,
                        })
            mlflow.log_params({"model/" + k: str(v)
                               for k, v in pipeline.named_steps["model"].get_params().items()
                               if v is not None})
            mlflow.log_params({"forecast_months": FORECAST_MONTHS,
                               "clip_at_zero": info["clip_predictions_at_zero"],
                               "prediction_kind": info["prediction_kind"],
                               "horizon_months": info["horizon_months"],
                               "output_units": "sales_units" if target_type == "sales" else "rental_unit_months"})
            forecast = ForecastModel(pipeline, list(numeric_features), list(categorical_features),
                                     FORECAST_MONTHS, info["clip_predictions_at_zero"])
            rows = xgb_predictions_pdf.loc[xgb_predictions_pdf["target_type"].eq(target_type)]
            check = rows.loc[rows["split"].eq("test")]
            np.testing.assert_allclose(forecast.predict(check[feature_columns]), check["prediction"],
                                       rtol=1e-5, atol=1e-5)
            example = check[feature_columns].head(5).copy()
            model_name = {"name" if "name" in inspect.signature(mlflow.sklearn.log_model).parameters
                          else "artifact_path": "forecast"}
            saved = mlflow.sklearn.log_model(
                sk_model=forecast, input_example=example,
                signature=infer_signature(example, forecast.predict(example)),
                serialization_format="cloudpickle", pip_requirements=requirements, **model_name,
            )
            restored = mlflow.pyfunc.load_model(saved.model_uri)
            np.testing.assert_allclose(np.asarray(restored.predict(example)).reshape(-1),
                                       forecast.predict(example), rtol=1e-5, atol=1e-5)
            mlflow.set_tag("reload_prediction_check", "passed")
            mlflow_forecast_models[target_type] = saved.model_uri
            overall = xgb_metrics_pdf.loc[xgb_metrics_pdf["target_type"].eq(target_type)]
            for estimator in ["xgboost", "training_mean"]:
                log_scores(overall.loc[overall["estimator"].eq(estimator)], f"test/{estimator}")
            log_frame(xgb_split_summary_pdf.loc[xgb_split_summary_pdf["target_type"].eq(target_type)], "coverage")
            run_records.append({"target_type": target_type, "mlflow_model_run_id": child.info.run_id,
                                "model_uri": saved.model_uri})

    mlflow_runs_pdf = pd.DataFrame(run_records)
    mlflow_runs_pdf["mlflow_experiment_id"] = MLFLOW_EXPERIMENT_ID
    mlflow_runs_pdf["mlflow_parent_run_id"] = MLFLOW_PARENT_RUN_ID
    run_ids = dict(zip(mlflow_runs_pdf["target_type"], mlflow_runs_pdf["mlflow_model_run_id"]))
    for name, frame in {
        "predictions": xgb_predictions_pdf, "metrics": xgb_metrics_pdf,
        "game_metrics": xgb_game_metrics_pdf,
        "split_summary": xgb_split_summary_pdf,
    }.items():
        frame["mlflow_experiment_id"] = MLFLOW_EXPERIMENT_ID
        frame["mlflow_parent_run_id"] = MLFLOW_PARENT_RUN_ID
        frame["mlflow_model_run_id"] = frame["target_type"].map(run_ids)
        log_frame(frame, name)
    tuning_mlflow_runs_pdf = pd.DataFrame(tuning_run_records)
    for name, frame in {
        "tuning_summary": tuning_summary_pdf, "tuning_trials": tuning_trials_pdf,
        "tuning_trial_folds": tuning_trial_folds_pdf, "tuning_folds": tuning_folds_pdf,
        "tuning_membership": tuning_membership_pdf, "tuning_runs": tuning_mlflow_runs_pdf,
    }.items():
        if not frame.empty:
            log_frame(frame, name)
    mlflow.log_dict(best_params_by_target, "selected_parameters.json")
    log_frame(mlflow_runs_pdf, "runs")
    log_frame(game_split_pdf.reset_index(), "game_split")
    if MLFLOW_LOG_DATA_SNAPSHOT:
        log_frame(model_pdf, "modeling_data")
    for (target_type, estimator), scores in xgb_metrics_pdf.groupby(["target_type", "estimator"]):
        log_scores(scores, f"{target_type}/{estimator}/test")
    mlflow.log_dict(mlflow_forecast_models, "forecast_model_uris.json")
    mlflow.log_text("import mlflow\n\n# Inputs: one row of cleaned numeric/categorical attributes per game.\n# Output: one total for the first six source calendar months per input row.\n"
                    + "\n".join(f"{target}_model = mlflow.pyfunc.load_model({uri!r})\n"
                                f"{target}_predictions = {target}_model.predict(clean_inputs)\n"
                                for target, uri in mlflow_forecast_models.items()), "reload_forecasts.py")

print(f"MLflow experiment: {experiment_path}")
print(f"Parent run: {MLFLOW_PARENT_RUN_ID}")
print("Forecast models:", mlflow_forecast_models)
display(mlflow_runs_pdf)


In [0]:
# CELL 7 — Six-month total errors and bracket accuracy on eligible TEST games
# Run after cell 4; run cell 6 first to attach these results to the MLflow runs.
# One game contributes one total for each eligible revenue type, with equal weight.
import time
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T

METRICS_TABLE = globals().get(
    "erro_table_path",
    "development_042_silver_sandbox.demand_forecast.forecast_error_metrics",
)
SAVE_METRICS = True
LOG_METRICS_TO_MLFLOW = True
BRACKET_ORDER = ["bottom_20", "middle_60", "top_20"]
IDENTITY = ["forecast_type", "metric_name", "revenue_type", "experiment_id", "experiment_name", "estimator"]
RUN_COLUMNS = ["mlflow_experiment_id", "mlflow_parent_run_id", "mlflow_model_run_id"]
GAME_KEYS = IDENTITY + ["game_key"]


def assign_total_bracket(values, lower, upper):
    """Apply the SAME actual-total thresholds to actual, predicted and baseline totals."""
    values = pd.Series(values, copy=False)
    if lower == upper:
        # A tied actual value belongs to the middle; ties are never split by name.
        return np.select([values.lt(lower), values.gt(upper)],
                         ["bottom_20", "top_20"], default="middle_60")
    return np.select([values.le(lower), values.gt(upper)],
                     ["bottom_20", "top_20"], default="middle_60")


def total_error_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    if not len(actual):
        return dict(mae=None, rmse=None, bias=None, wape=None, smape=None, r2=None)
    error = predicted - actual
    absolute_error = np.abs(error)
    denominator = np.abs(actual).sum()
    symmetric_denominator = np.abs(actual) + np.abs(predicted)
    smape_terms = np.divide(2 * absolute_error, symmetric_denominator,
                            out=np.zeros_like(error), where=symmetric_denominator > 0)
    sst = np.square(actual - actual.mean()).sum()
    return {
        "mae": float(absolute_error.mean()),
        "rmse": float(np.sqrt(np.square(error).mean())),
        "bias": float(error.mean()),
        "wape": float(absolute_error.sum() / denominator) if denominator > 0 else None,
        "smape": float(smape_terms.mean()),
        "r2": float(1 - np.square(error).sum() / sst) if len(actual) > 1 and sst > 0 else None,
    }


def build_bracket_metrics(predictions):
    """Completed six-month targets only; thresholds use held-out actuals for evaluation."""
    required = {
        "game_key", "target_type", "experiment_id", "experiment_name", "prediction",
        "baseline_prediction", "target_value", "split", "used_for_test", "is_observed",
        "target_is_imputed", "has_complete_horizon", "coverage_months",
        "target_window_end_month", "target_available_date",
    }
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Missing total-prediction columns: {sorted(missing)}. Run cell 4 first.")
    selected = (
        predictions["split"].eq("test")
        & predictions["used_for_test"].eq(True).fillna(False)
        & predictions["is_observed"].eq(True).fillna(False)
        & predictions["has_complete_horizon"].eq(True).fillna(False)
        & predictions["target_is_imputed"].eq(False).fillna(False)
        & pd.to_numeric(predictions["coverage_months"], errors="coerce").ge(FORECAST_MONTHS)
        & predictions["target_type"].isin(["sales", "rental"])
    )
    flag_columns = [
        "actual_bracket", "predicted_bracket", "baseline_bracket", "bracket_correct",
        "baseline_bracket_correct", "bracket_lower_cutoff", "bracket_upper_cutoff",
        "cohort_test_games", "bracket_actual", "bracket_month_index", "bracket_rank",
    ]
    frame = predictions.loc[selected].drop(columns=flag_columns, errors="ignore").copy()
    for name in ["target_value", "prediction", "baseline_prediction"]:
        frame[name] = pd.to_numeric(frame[name], errors="coerce")
    if frame.empty:
        raise ValueError("No test games have an eligible six-month total; nothing was written.")
    if not np.isfinite(frame[["target_value", "prediction", "baseline_prediction"]].to_numpy()).all():
        raise ValueError("Eligible test totals, predictions and baselines must all be finite.")
    for name in ["game_key", "experiment_id", "experiment_name"]:
        if frame[name].isna().any() or frame[name].astype(str).str.strip().eq("").any():
            raise ValueError(f"Missing identity value in {name}.")
        frame[name] = frame[name].astype(str)
    frame["forecast_type"] = "demand"
    frame["metric_name"] = frame["target_type"].map({
        "sales": f"total_sales_{FORECAST_MONTHS}m",
        "rental": f"total_rental_{FORECAST_MONTHS}m",
    })
    frame["revenue_type"] = frame["target_type"].map({"sales": "sales", "rental": "lease"})
    frame["estimator"] = "xgboost"
    frame["target_game"] = frame["game_key"]
    frame["actual_total"] = frame["target_value"]
    frame["predicted_total"] = frame["prediction"]
    frame["baseline_total"] = frame["baseline_prediction"]
    for name in RUN_COLUMNS:
        if name not in frame:
            frame[name] = None
    if frame.duplicated(GAME_KEYS).any():
        raise ValueError("Duplicate six-month totals for a game/model. Metrics were not written.")
    for _, group in frame.groupby(IDENTITY, sort=False):
        for name in RUN_COLUMNS:
            if group[name].dropna().nunique() > 1:
                raise ValueError(f"Multiple {name} values in one model/experiment.")
    frame = frame.sort_values(GAME_KEYS).reset_index(drop=True)

    for _, group in frame.groupby(IDENTITY, sort=False):
        lower = float(group["actual_total"].quantile(0.20))
        upper = float(group["actual_total"].quantile(0.80))
        index = group.index
        frame.loc[index, "bracket_lower_cutoff"] = lower
        frame.loc[index, "bracket_upper_cutoff"] = upper
        frame.loc[index, "cohort_test_games"] = len(group)
        for value_column, bracket_column in [
            ("actual_total", "actual_bracket"), ("predicted_total", "predicted_bracket"),
            ("baseline_total", "baseline_bracket"),
        ]:
            frame.loc[index, bracket_column] = assign_total_bracket(group[value_column], lower, upper)
    frame["bracket_correct"] = frame["actual_bracket"].eq(frame["predicted_bracket"])
    frame["baseline_bracket_correct"] = frame["actual_bracket"].eq(frame["baseline_bracket"])
    frame["error"] = frame["predicted_total"] - frame["actual_total"]
    frame["absolute_error"] = frame["error"].abs()
    frame["squared_error"] = frame["error"] ** 2
    frame["cohort_test_games"] = frame["cohort_test_games"].astype(int)

    output = []
    for level, group_columns in [
        ("per_game", GAME_KEYS), ("overall", IDENTITY), ("bracket", IDENTITY + ["actual_bracket"]),
    ]:
        for key, group in frame.groupby(group_columns, sort=False):
            row = dict(zip(group_columns, key))
            row.update(total_error_metrics(group["actual_total"], group["predicted_total"]))
            row.update({f"baseline_{k}": v for k, v in
                        total_error_metrics(group["actual_total"], group["baseline_total"]).items()})
            is_game = level == "per_game"
            row.update({
                "result_level": level,
                "bracket": "all_test_games" if level == "overall" else group["actual_bracket"].iloc[0],
                "game_key": group["game_key"].iloc[0] if is_game else None,
                "target_game": group["target_game"].iloc[0] if is_game else None,
                "test_games": int(len(group)),
                "scored_totals": int(len(group)),
                "horizon_months": int(FORECAST_MONTHS),
                # Compatibility fields identify one six-month window, not six predictions.
                "valid_months": int(FORECAST_MONTHS * len(group)),
                "first_scored_month": 1, "last_scored_month": int(FORECAST_MONTHS),
                "directional_comparisons": 0, "directional_accuracy": None,
                "bracket_accuracy": float(group["bracket_correct"].mean()),
                "baseline_bracket_accuracy": float(group["baseline_bracket_correct"].mean()),
                "correct_bracket_games": int(group["bracket_correct"].sum()),
                "baseline_correct_bracket_games": int(group["baseline_bracket_correct"].sum()),
                "bracket_lower_cutoff": float(group["bracket_lower_cutoff"].iloc[0]),
                "bracket_upper_cutoff": float(group["bracket_upper_cutoff"].iloc[0]),
                "cohort_test_games": int(group["cohort_test_games"].iloc[0]),
                "bracket_basis": "test_actual_total_20th_80th_percentiles",
                "evaluation_split": "test",
            })
            for name in ["actual_total", "predicted_total", "baseline_total", "coverage_months",
                         "has_complete_horizon", "target_window_end_month", "target_available_date",
                         "actual_bracket", "predicted_bracket", "baseline_bracket", "bracket_correct",
                         "baseline_bracket_correct"]:
                row[name] = group[name].iloc[0] if is_game else None
            row["bracket_actual"] = row["actual_total"]  # Older consumers of the shared metrics table.
            row["bracket_month_index"] = int(FORECAST_MONTHS) if is_game else None
            for name in RUN_COLUMNS:
                nonmissing = group[name].dropna()
                row[name] = str(nonmissing.iloc[0]) if len(nonmissing) else None
            output.append(row)
    metrics = pd.DataFrame(output)
    bracket_columns = list(dict.fromkeys(
        GAME_KEYS + ["target_type", "target_game", "actual_total", "predicted_total", "baseline_total",
                     "actual_bracket", "predicted_bracket", "baseline_bracket", "bracket_correct",
                     "baseline_bracket_correct", "bracket_lower_cutoff", "bracket_upper_cutoff",
                     "cohort_test_games", "coverage_months", "target_window_end_month", "target_available_date"]
    ))
    brackets = frame[bracket_columns].copy()
    confusion_rows = []
    for key, group in frame.groupby(IDENTITY, sort=False):
        for predictor, predicted_column in [("xgboost", "predicted_bracket"), ("training_mean", "baseline_bracket")]:
            for actual_bracket in BRACKET_ORDER:
                actual_group = group.loc[group["actual_bracket"].eq(actual_bracket)]
                for predicted_bracket in BRACKET_ORDER:
                    count = int(actual_group[predicted_column].eq(predicted_bracket).sum())
                    confusion_rows.append({
                        **dict(zip(IDENTITY, key)), "predictor": predictor,
                        "actual_bracket": actual_bracket, "predicted_bracket": predicted_bracket,
                        "games": count,
                        "fraction_of_actual_bracket": count / len(actual_group) if len(actual_group) else None,
                    })
    return frame, brackets, metrics, pd.DataFrame(confusion_rows)


evaluation_rows_pdf, game_brackets_pdf, forecast_error_metrics_pdf, bracket_confusion_pdf = build_bracket_metrics(
    xgb_predictions_pdf)

# An explicit Spark schema supports all-null aggregate fields and unlogged MLflow IDs.
string_cols = ["result_level", *IDENTITY, "bracket", "game_key", "target_game",
               "bracket_basis", "evaluation_split", "actual_bracket", "predicted_bracket",
               "baseline_bracket", *RUN_COLUMNS]
long_cols = ["test_games", "scored_totals", "horizon_months", "valid_months", "directional_comparisons",
             "first_scored_month", "last_scored_month", "bracket_month_index", "cohort_test_games",
             "correct_bracket_games", "baseline_correct_bracket_games", "coverage_months"]
double_cols = ["mae", "rmse", "bias", "wape", "smape", "r2", "baseline_mae", "baseline_rmse",
               "baseline_bias", "baseline_wape", "baseline_smape", "baseline_r2", "directional_accuracy",
               "bracket_accuracy", "baseline_bracket_accuracy", "bracket_actual", "actual_total",
               "predicted_total", "baseline_total", "bracket_lower_cutoff", "bracket_upper_cutoff"]
bool_cols = ["has_complete_horizon", "bracket_correct", "baseline_bracket_correct"]
date_cols = ["target_window_end_month", "target_available_date"]
output_columns = string_cols + long_cols + double_cols + bool_cols + date_cols
schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in string_cols]
    + [T.StructField(c, T.LongType(), True) for c in long_cols]
    + [T.StructField(c, T.DoubleType(), True) for c in double_cols]
    + [T.StructField(c, T.BooleanType(), True) for c in bool_cols]
    + [T.StructField(c, T.DateType(), True) for c in date_cols])
records = []
for row in forecast_error_metrics_pdf[output_columns].to_dict("records"):
    records.append(tuple(
        None if pd.isna(row[c]) else
        str(row[c]) if c in string_cols else int(row[c]) if c in long_cols else
        float(row[c]) if c in double_cols else bool(row[c]) if c in bool_cols else
        pd.Timestamp(row[c]).date()
        for c in output_columns))
forecast_error_metrics_df = spark.createDataFrame(records, schema).withColumn("created_at", F.current_timestamp())
per_game_metrics_df = forecast_error_metrics_df.where(F.col("result_level") == "per_game")
overall_metrics_df = forecast_error_metrics_df.where(F.col("result_level") == "overall")
bracket_metrics_df = forecast_error_metrics_df.where(F.col("result_level") == "bracket")

# Attach actual/predicted bracket and match flags to each evaluated game/model total.
# Unlabeled, training, and incomplete-horizon games retain null evaluation brackets.
join_keys = ["experiment_id", "experiment_name", "target_type", "game_key"]
flag_columns = ["actual_bracket", "predicted_bracket", "baseline_bracket", "bracket_correct",
                "baseline_bracket_correct", "bracket_lower_cutoff", "bracket_upper_cutoff", "cohort_test_games"]
xgb_predictions_pdf = xgb_predictions_pdf.drop(columns=flag_columns, errors="ignore").merge(
    game_brackets_pdf[join_keys + flag_columns], on=join_keys, how="left", validate="one_to_one")

if SAVE_METRICS:
    # Reruns replace only this experiment's metrics; other experiments remain intact.
    experiments = sorted(forecast_error_metrics_pdf["experiment_id"].unique())
    predicate = "experiment_id IN (" + ",".join("'" + e.replace("'", "''") + "'" for e in experiments) + ")"
    writer = forecast_error_metrics_df.write.format("delta")
    if spark.catalog.tableExists(METRICS_TABLE):
        (writer.mode("overwrite").option("replaceWhere", predicate)
         .option("mergeSchema", "true").saveAsTable(METRICS_TABLE))
    else:
        writer.mode("errorifexists").saveAsTable(METRICS_TABLE)
    print(f"Saved six-month-total metrics to {METRICS_TABLE}; replaced experiment(s): {experiments}")

if LOG_METRICS_TO_MLFLOW:
    from mlflow.tracking import MlflowClient
    from mlflow.entities import Metric
    client = MlflowClient()
    linked = forecast_error_metrics_pdf.dropna(subset=["mlflow_model_run_id", "mlflow_parent_run_id"])
    if linked.empty:
        print("Run cell 6, then rerun cell 7 to attach evaluation to MLflow.")
    else:
        timestamp = int(time.time() * 1000)
        summary = linked.loc[linked["result_level"].isin(["overall", "bracket"])]
        batches = {}
        metric_names = ["mae", "rmse", "bias", "wape", "smape", "r2", "baseline_mae", "baseline_rmse",
                        "baseline_bias", "baseline_wape", "baseline_smape", "baseline_r2",
                        "bracket_accuracy", "baseline_bracket_accuracy", "correct_bracket_games",
                        "baseline_correct_bracket_games", "test_games", "scored_totals"]
        for row in summary.to_dict("records"):
            suffix = f"test_total_by_actual/{row['result_level']}/{row['bracket']}"
            for run_id, prefix in [(row["mlflow_model_run_id"], suffix),
                                   (row["mlflow_parent_run_id"], row["revenue_type"] + "/" + suffix)]:
                bucket = batches.setdefault(run_id, [])
                for metric in metric_names:
                    value = row[metric]
                    if pd.notna(value) and np.isfinite(value):
                        bucket.append(Metric(f"{prefix}/{metric}", float(value), timestamp, 0))
        for run_id, batch in batches.items():
            # Keep requests under MLflow's per-batch metric limit.
            for offset in range(0, len(batch), 500):
                client.log_batch(run_id, metrics=batch[offset:offset + 500])
        with tempfile.TemporaryDirectory(prefix="xgb_total_brackets_") as directory:
            for run_id, rows in linked.groupby("mlflow_parent_run_id"):
                experiments = set(rows["experiment_id"])
                artifacts = {
                    "test_bracket_and_game_metrics.csv": rows[output_columns],
                    "game_brackets.csv": game_brackets_pdf.loc[game_brackets_pdf["experiment_id"].isin(experiments)],
                    "bracket_confusion.csv": bracket_confusion_pdf.loc[bracket_confusion_pdf["experiment_id"].isin(experiments)],
                }
                for filename, data in artifacts.items():
                    path = Path(directory) / filename
                    data.to_csv(path, index=False)
                    client.log_artifact(run_id, str(path), artifact_path="evaluation")
        print("Logged total errors, bracket accuracy, per-game matches and confusion matrices to MLflow.")

print(f"Each row scores one {FORECAST_MONTHS}-month total; every eligible test game has equal weight.")
print("Bracket accuracy = fraction of games whose predicted total falls in its actual-total bracket.")
print("The test actual 20th/80th percentiles define common cutoffs for actuals, predictions and the training-mean baseline.")
print("Ties stay together, so brackets may not contain exactly 20% / 60% / 20%; equal cutoffs place tied totals in the middle.")
print("These cutoffs are retrospective evaluation thresholds; they are not learned model inputs or deployment thresholds.")
print("Bracket accuracy is 0–1; WAPE is a fraction and may exceed 1; sMAPE is 0–2. Directional accuracy is undefined for one total.")
display(forecast_error_metrics_df.orderBy("experiment_id", "revenue_type", "result_level", "bracket", "target_game"))
display(game_brackets_pdf.sort_values(["revenue_type", "actual_total"], ascending=[True, False]))
display(bracket_confusion_pdf)
